# Arena GA Evolution — 戦略進化アリーナ
遺伝的アルゴリズムで戦略を進化させる。毎週:
1. 全10個体の週次P&LをBQから取得
2. 適応度順にソート
3. 上位3個体を選抜 → 交叉 → 突然変異
4. 下位3個体を新個体で置換
5. 新世代をBQにINSERT

In [ ]:
# 設定
PROJECT_ID = 'your-gcp-project'
GENE_TABLE = f'{PROJECT_ID}.stock_gold.ga_gene_pool'
FITNESS_VIEW = f'''
  SELECT gene_id, strategy_name, generation,
         SUM(week_pnl) AS cumulative_pnl,
         AVG(win_rate) AS avg_win_rate,
         COUNT(*) AS weeks_alive
  FROM `{PROJECT_ID}.stock_gold.ga_weekly_pnl`
  WHERE week_start >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
  GROUP BY gene_id, strategy_name, generation
  ORDER BY cumulative_pnl DESC
'''

In [ ]:
import pandas as pd
import numpy as np
from google.colab import auth
auth.authenticate_user()

# 現世代の適応度を取得
df_fitness = pd.read_gbq(FITNESS_VIEW, project_id=PROJECT_ID)
df_fitness

In [ ]:
# 現世代の遺伝子を取得
df_genes = pd.read_gbq(f'''
  SELECT * FROM `{GENE_TABLE}`
  WHERE generation = (SELECT MAX(generation) FROM `{GENE_TABLE}`)
''', project_id=PROJECT_ID)
df_genes = df_genes.merge(df_fitness[['gene_id','cumulative_pnl']], on='gene_id', how='left')
df_genes = df_genes.sort_values('cumulative_pnl', ascending=False)
print(f'現世代: gen={df_genes.generation.max()}, 個体数={len(df_genes)}')
df_genes[['gene_id','strategy_name','cumulative_pnl']]

In [ ]:
# ============================================================
# GA コアロジック
# ============================================================

GENE_COLS = ['w_momentum','w_volume','w_reversal','w_breakout',
             'w_dow','w_monthend','w_sentiment','w_mean_rev',
             'threshold','max_positions','stop_loss','take_profit','max_hold_days']

SELECTION_SIZE = 3   # 上位3が親
MUTATION_RATE = 0.1  # 突然変異率
MUTATION_STD = 0.15  # 突然変異幅
POP_SIZE = 10        # 固定

def select_parents(df, n=SELECTION_SIZE):
    """適応度順に上位n個体を選抜"""
    return df.nlargest(n, 'cumulative_pnl')

def crossover(p1, p2):
    """一様交叉: 各遺伝子を50%の確率でどちらかの親から継承"""
    child = {}
    for col in GENE_COLS:
        if col in ['max_positions', 'max_hold_days']:
            # 整数値はランダムに選択
            child[col] = np.random.choice([p1[col], p2[col]])
        else:
            # 実数値はブレンド
            alpha = np.random.random()
            child[col] = p1[col] * alpha + p2[col] * (1 - alpha)
    return child

def mutate(genes):
    """突然変異: 各遺伝子をMUTATION_RATEの確率で摂動"""
    for col in GENE_COLS:
        if np.random.random() < MUTATION_RATE:
            if col in ['max_positions', 'max_hold_days']:
                # 整数値 = ±1 or ±2
                delta = int(np.random.choice([-2, -1, 1, 2]))
                genes[col] = max(1, min(col == 'max_positions' and 3 or 21,
                                        genes[col] + delta))
            else:
                # 実数値 = 正規分布ノイズ
                noise = np.random.normal(0, MUTATION_STD)
                if col in ['w_momentum','w_volume','w_reversal','w_breakout',
                           'w_dow','w_monthend','w_sentiment','w_mean_rev']:
                    genes[col] = np.clip(genes[col] + noise, 0, 1)
                elif col == 'threshold':
                    genes[col] = np.clip(genes[col] + noise, 0.1, 2.0)
                elif col == 'stop_loss':
                    genes[col] = np.clip(genes[col] + noise, 0.5, 10.0)
                elif col == 'take_profit':
                    genes[col] = np.clip(genes[col] + noise, 1.0, 15.0)
    return genes

def next_generation(df_current):
    """次世代を生成: 選抜→交叉→突然変異"""
    current_gen = int(df_current.generation.max())
    parents = select_parents(df_current)
    
    new_genes = []
    # Elite: 上位2個体をそのまま残す
    for i in range(2):
        genes = parents.iloc[i].to_dict()
        genes['generation'] = current_gen + 1
        genes['parent_a'] = genes['gene_id']
        genes['parent_b'] = None
        genes['gene_id'] = int(df_current['gene_id'].max()) + i + 1
        genes['strategy_name'] = f'GA_gen{current_gen+1}_elite_{i+1}'
        genes['fitness'] = None
        genes['n_trades'] = 0
        genes['win_rate'] = None
        genes['weekly_pnl'] = None
        new_genes.append(genes)
    
    # 交叉: 上位3からランダムペアで3個体生成
    for i in range(3):
        p1 = parents.sample(1).iloc[0]
        p2 = parents.sample(1).iloc[0]
        child = crossover(p1, p2)
        child = mutate(child)
        child['generation'] = current_gen + 1
        child['parent_a'] = int(p1['gene_id'])
        child['parent_b'] = int(p2['gene_id'])
        child['gene_id'] = int(df_current['gene_id'].max()) + 2 + i + 1
        child['strategy_name'] = f'GA_gen{current_gen+1}_child_{i+1}'
        child['fitness'] = None
        child['n_trades'] = 0
        child['win_rate'] = None
        child['weekly_pnl'] = None
        new_genes.append(child)
    
    # 突然変異個体: 上位から変異で2個体
    for i in range(2):
        parent = parents.sample(1).iloc[0]
        mutant = parent.to_dict()
        for col in GENE_COLS:
            mutant[col] = parent[col]
        mutant = mutate(mutant)
        mutant['generation'] = current_gen + 1
        mutant['parent_a'] = int(parent['gene_id'])
        mutant['parent_b'] = None
        mutant['gene_id'] = int(df_current['gene_id'].max()) + 5 + i + 1
        mutant['strategy_name'] = f'GA_gen{current_gen+1}_mutant_{i+1}'
        mutant['fitness'] = None
        mutant['n_trades'] = 0
        mutant['win_rate'] = None
        mutant['weekly_pnl'] = None
        new_genes.append(mutant)
    
    # 残り枠: 完全ランダム
    remaining = POP_SIZE - len(new_genes)
    for i in range(remaining):
        new_genes.append({
            'gene_id': int(df_current['gene_id'].max()) + 7 + i + 1,
            'generation': current_gen + 1,
            'parent_a': None, 'parent_b': None,
            'strategy_name': f'GA_gen{current_gen+1}_random_{i+1}',
            'w_momentum': np.random.random(),
            'w_volume': np.random.random(),
            'w_reversal': np.random.random() * 0.8,
            'w_breakout': np.random.random() * 0.7,
            'w_dow': np.random.random() * 0.4,
            'w_monthend': np.random.random() * 0.4,
            'w_sentiment': np.random.random() * 0.6,
            'w_mean_rev': np.random.random() * 0.5,
            'threshold': 0.2 + np.random.random() * 0.8,
            'max_positions': int(1 + np.random.random() * 2),
            'stop_loss': 2 + np.random.random() * 5,
            'take_profit': 3 + np.random.random() * 7,
            'max_hold_days': int(3 + np.random.random() * 12),
            'fitness': None, 'n_trades': 0,
            'win_rate': None, 'weekly_pnl': None,
            'created_at': pd.Timestamp.now()
        })
    
    return pd.DataFrame(new_genes)

# 次世代を生成
df_next = next_generation(df_genes)
df_next[['gene_id','generation','strategy_name']]

In [ ]:
# 次世代をBQに書き込み
# 注意: gene_idの重複防止のため、現在のMAXを事前確認

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# 現在の最大gene_idを取得
max_id = client.query(f'SELECT MAX(gene_id) FROM `{GENE_TABLE}`').result().to_dataframe().iloc[0,0]
print(f'現在の最大gene_id: {max_id}')

# gene_idを再割り当て
df_next['gene_id'] = range(max_id + 1, max_id + 1 + len(df_next))
df_next['created_at'] = pd.Timestamp.now()

# BQに書き込み
job = client.load_table_from_dataframe(df_next, GENE_TABLE)
job.result()
print(f'新世代{len(df_next)}個体をINSERT完了')

In [ ]:
# 進化の可視化
import matplotlib.pyplot as plt

# 全世代のデータ取得
df_all = client.query(f'''
  SELECT w.gene_id, g.generation, g.strategy_name,
         w.week_start, w.week_pnl, w.win_rate
  FROM `{PROJECT_ID}.stock_gold.ga_weekly_pnl` w
  JOIN `{GENE_TABLE}` g USING(gene_id)
  ORDER BY w.week_start, w.week_pnl DESC
''').result().to_dataframe()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 左: 世代ごとのfitness分布
for gen in sorted(df_all.generation.unique()):
    data = df_all[df_all.generation == gen].groupby('gene_id')['week_pnl'].sum()
    axes[0].bar(gen, data.mean(), yerr=data.std(), alpha=0.6)
axes[0].set_xlabel('Generation')
axes[0].set_ylabel('Avg Cumulative P&L')
axes[0].set_title('Fitness Evolution')

# 中: 最新週のTop5戦略
latest = df_all[df_all.week_start == df_all.week_start.max()]
top5 = latest.groupby('strategy_name')['week_pnl'].sum().nlargest(5)
axes[1].barh(range(len(top5)), top5.values)
axes[1].set_yticks(range(len(top5)))
axes[1].set_yticklabels(top5.index)
axes[1].set_title(f'Top 5 — Week {df_all.week_start.max()}')

# 右: 遺伝子多様性 (重みの分散)
latest_genes = df_all[df_all.week_start == df_all.week_start.max()]
weight_cols = ['w_momentum','w_volume','w_reversal','w_breakout']
for i, col in enumerate(weight_cols):
    vals = latest_genes.groupby('gene_id')[col].mean()
    axes[2].bar(i, vals.std(), alpha=0.6)
axes[2].set_xticks(range(len(weight_cols)))
axes[2].set_xticklabels(['Mom','Vol','Rev','Brk'], rotation=15)
axes[2].set_title('Gene Diversity (std)')

plt.tight_layout()
plt.show()

In [ ]:
# 週次実行: 1クリックで全行程
# ga_gene_pool → ga_signals → ga_positions → ga_weekly_pnl → GA進化

!bq query --use_legacy_sql=false < gold/arena_ga.sql